In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed.csv")
df["text"] = df["text"].fillna("")

X = df.drop(columns="fraudulent")
y = df["fraudulent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Fake % in train:", round(y_train.mean() * 100, 2))
print("Fake % in test: ", round(y_test.mean() * 100, 2))

Train: (12524, 13)  Test: (3132, 13)
Fake % in train: 4.52
Fake % in test:  4.5


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    sublinear_tf=True,
)

X_train_text = tfidf.fit_transform(X_train["text"])
X_test_text = tfidf.transform(X_test["text"])

print("Train matrix:", X_train_text.shape)
print("Test matrix: ", X_test_text.shape)
print("Some words learned:", tfidf.get_feature_names_out()[1000:1010])

Train matrix: (12524, 50000)
Test matrix:  (3132, 50000)
Some words learned: ['advancement' 'advancement chosen' 'advancement current'
 'advancement department' 'advancement extensive' 'advancement growth'
 'advancement mileage' 'advancement opportunities' 'advancement training'
 'advancement valid']


In [12]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

nb = MultinomialNB()
nb.fit(X_train_text, y_train)

y_pred = nb.predict(X_test_text)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, digits=3, target_names=["real", "fake"]))

Confusion matrix:
[[2990    1]
 [ 104   37]]

              precision    recall  f1-score   support

        real      0.966     1.000     0.983      2991
        fake      0.974     0.262     0.413       141

    accuracy                          0.966      3132
   macro avg      0.970     0.631     0.698      3132
weighted avg      0.967     0.966     0.957      3132



In [13]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight="balanced", max_iter=2000)
lr.fit(X_train_text, y_train)

y_pred_lr = lr.predict(X_test_text)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print()
print(classification_report(y_test, y_pred_lr, digits=3, target_names=["real", "fake"]))

Confusion matrix:
[[2945   46]
 [  16  125]]

              precision    recall  f1-score   support

        real      0.995     0.985     0.990      2991
        fake      0.731     0.887     0.801       141

    accuracy                          0.980      3132
   macro avg      0.863     0.936     0.895      3132
weighted avg      0.983     0.980     0.981      3132



In [14]:
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack, csr_matrix

meta_cols = [c for c in X_train.columns if c != "text"]

scaler = MinMaxScaler()
X_train_meta = scaler.fit_transform(X_train[meta_cols])
X_test_meta = scaler.transform(X_test[meta_cols])

X_train_all = hstack([X_train_text, csr_matrix(X_train_meta)]).tocsr()
X_test_all = hstack([X_test_text, csr_matrix(X_test_meta)]).tocsr()

print("Meta columns:", len(meta_cols))
print("Combined train:", X_train_all.shape)
print("Combined test: ", X_test_all.shape)

lr_all = LogisticRegression(class_weight="balanced", max_iter=2000)
lr_all.fit(X_train_all, y_train)

y_pred_all = lr_all.predict(X_test_all)

print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_all))
print()
print(classification_report(y_test, y_pred_all, digits=3, target_names=["real", "fake"]))

Meta columns: 12
Combined train: (12524, 50012)
Combined test:  (3132, 50012)

Confusion matrix:
[[2877  114]
 [  13  128]]

              precision    recall  f1-score   support

        real      0.996     0.962     0.978      2991
        fake      0.529     0.908     0.668       141

    accuracy                          0.959      3132
   macro avg      0.762     0.935     0.823      3132
weighted avg      0.974     0.959     0.964      3132



In [15]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

models = {
    "Naive Bayes (text)": (nb, X_test_text),
    "LogReg balanced (text)": (lr, X_test_text),
    "LogReg balanced (text + meta)": (lr_all, X_test_all),
}

rows = []
for name, (model, X_te) in models.items():
    prob = model.predict_proba(X_te)[:, 1]
    pred = (prob >= 0.5).astype(int)
    rows.append({
        "model": name,
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "pr_auc": average_precision_score(y_test, prob),
    })

results = pd.DataFrame(rows).round(3)
results

,model,precision,recall,f1,pr_auc
0,Naive Bayes (text),0.974,0.262,0.413,0.471
1,LogReg balanced (text),0.731,0.887,0.801,0.904
2,LogReg balanced (text + meta),0.529,0.908,0.668,0.851


In [16]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import precision_recall_curve

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_prob = cross_val_predict(
    LogisticRegression(class_weight="balanced", max_iter=2000),
    X_train_text, y_train,
    cv=cv, method="predict_proba",
)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, oof_prob)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best = np.argmax(f1[:-1])
best_threshold = thresholds[best]

print("Best threshold:", round(best_threshold, 3))
print("Cross-val  precision:", round(precision[best], 3),
      " recall:", round(recall[best], 3),
      " F1:", round(f1[best], 3))

test_prob = lr.predict_proba(X_test_text)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

print()
print("Test set with tuned threshold:")
print(confusion_matrix(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, digits=3, target_names=["real", "fake"]))

Best threshold: 0.552
Cross-val  precision: 0.792  recall: 0.788  F1: 0.79

Test set with tuned threshold:
[[2963   28]
 [  18  123]]

              precision    recall  f1-score   support

        real      0.994     0.991     0.992      2991
        fake      0.815     0.872     0.842       141

    accuracy                          0.985      3132
   macro avg      0.904     0.931     0.917      3132
weighted avg      0.986     0.985     0.986      3132



In [17]:
feature_names = tfidf.get_feature_names_out()
weights = lr.coef_[0]

order = np.argsort(weights)
top_fake_idx = order[-20:][::-1]
top_real_idx = order[:20]

top_words = pd.DataFrame({
    "red flag word": feature_names[top_fake_idx],
    "fake weight": weights[top_fake_idx].round(2),
    "trust word": feature_names[top_real_idx],
    "real weight": weights[top_real_idx].round(2),
})
top_words

,red flag word,fake weight,trust word,real weight
0,link urltoken,4.16,english,-2.49
1,using link,3.78,growing,-2.49
2,apply using,3.62,digital,-2.34
3,assistant,3.44,companies,-2.25
4,administrative,2.95,new,-2.24
5,moneytoken,2.95,clients,-2.12
6,receptionist,2.83,team,-2.07
7,hospital,2.80,web,-2.03
8,link,2.79,client,-2.01
9,subsea,2.71,recruitment,-1.98


In [18]:
train_texts = set(X_train["text"])
is_dup = X_test["text"].isin(train_texts).values

print("Test postings with an exact copy in train:", is_dup.sum(), "of", len(X_test))
print("Of those, fake:", y_test[is_dup].sum())
print()

print("Scores on test postings WITHOUT a copy in train:")
print(classification_report(
    y_test[~is_dup], test_pred[~is_dup],
    digits=3, target_names=["real", "fake"],
))

Test postings with an exact copy in train: 0 of 3132
Of those, fake: 0

Scores on test postings WITHOUT a copy in train:
              precision    recall  f1-score   support

        real      0.994     0.991     0.992      2991
        fake      0.815     0.872     0.842       141

    accuracy                          0.985      3132
   macro avg      0.904     0.931     0.917      3132
weighted avg      0.986     0.985     0.986      3132



In [19]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000)),
])

param_grid = {"clf__C": [0.5, 1, 2, 4, 8, 16]}

search = GridSearchCV(
    pipe,
    param_grid,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train["text"], y_train)

print("Best C:", search.best_params_["clf__C"])
print("Best cross-val PR-AUC:", round(search.best_score_, 3))

pd.DataFrame(search.cv_results_)[
    ["param_clf__C", "mean_test_score", "std_test_score"]
].round(3)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best C: 16
Best cross-val PR-AUC: 0.882


,param_clf__C,mean_test_score,std_test_score
0,0.5,0.840,0.020
1,1.0,0.859,0.016
2,2.0,0.867,0.015
3,4.0,0.875,0.015
4,8.0,0.880,0.015
5,16.0,0.882,0.015


In [ ]:
import time
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


def make_tfidf():
    return TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )


candidates = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=2000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
    "Neural Network (MLP)": MLPClassifier(
        hidden_layer_sizes=(64,),
        early_stopping=True,
        max_iter=50,
        random_state=42,
    ),
}

rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("tfidf", make_tfidf()), ("clf", clf)])
    start = time.time()
    scores = cross_val_score(pipe, X_train["text"], y_train, cv=cv, scoring="average_precision")
    rows.append({
        "model": name,
        "cv_pr_auc": scores.mean(),
        "std": scores.std(),
        "seconds": time.time() - start,
    })
    print(f"{name}: done in {time.time() - start:.0f}s")

comparison = pd.DataFrame(rows).round(3)
comparison

In [21]:
import time
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


def make_tfidf():
    return TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
        sublinear_tf=True,
    )


candidates = {
    "Logistic Regression": LogisticRegression(C=8, class_weight="balanced", max_iter=2000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
    "Neural Network (MLP)": MLPClassifier(
        hidden_layer_sizes=(64,),
        early_stopping=True,
        max_iter=50,
        random_state=42,
    ),
}

rows = []
for name, clf in candidates.items():
    pipe = Pipeline([("tfidf", make_tfidf()), ("clf", clf)])
    start = time.time()
    scores = cross_val_score(pipe, X_train["text"], y_train, cv=cv, scoring="average_precision")
    rows.append({
        "model": name,
        "cv_pr_auc": scores.mean(),
        "std": scores.std(),
        "seconds": time.time() - start,
    })
    print(f"{name}: done in {time.time() - start:.0f}s")

comparison = pd.DataFrame(rows).round(3)
comparison

Logistic Regression: done in 62s
Random Forest: done in 67s
Neural Network (MLP): done in 708s


,model,cv_pr_auc,std,seconds
0,Logistic Regression,0.880,0.015,62.076
1,Random Forest,0.826,0.033,66.592
2,Neural Network (MLP),0.878,0.009,708.070


In [22]:
final_pipe = Pipeline([
    ("tfidf", make_tfidf()),
    ("clf", LogisticRegression(C=8, class_weight="balanced", max_iter=2000)),
])

oof_prob = cross_val_predict(
    final_pipe, X_train["text"], y_train,
    cv=cv, method="predict_proba",
)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_train, oof_prob)
f1 = 2 * precision * recall / (precision + recall + 1e-12)
best = np.argmax(f1[:-1])
final_threshold = float(thresholds[best])

print("Final threshold:", round(final_threshold, 3))
print("Cross-val  precision:", round(precision[best], 3),
      " recall:", round(recall[best], 3),
      " F1:", round(f1[best], 3))

final_pipe.fit(X_train["text"], y_train)

final_prob = final_pipe.predict_proba(X_test["text"])[:, 1]
final_pred = (final_prob >= final_threshold).astype(int)

print()
print("FINAL TEST RESULTS")
print(confusion_matrix(y_test, final_pred))
print()
print(classification_report(y_test, final_pred, digits=3, target_names=["real", "fake"]))
print("Test PR-AUC:", round(average_precision_score(y_test, final_prob), 3))

Final threshold: 0.448
Cross-val  precision: 0.833  recall: 0.809  F1: 0.821

FINAL TEST RESULTS
[[2961   30]
 [  18  123]]

              precision    recall  f1-score   support

        real      0.994     0.990     0.992      2991
        fake      0.804     0.872     0.837       141

    accuracy                          0.985      3132
   macro avg      0.899     0.931     0.914      3132
weighted avg      0.985     0.985     0.985      3132

Test PR-AUC: 0.908


In [24]:
import joblib
from pathlib import Path

In [25]:
bundle = {
    "pipeline": final_pipe,
    "threshold": final_threshold,
    "metrics": {
        "precision": round(precision_score(y_test, final_pred), 3),
        "recall": round(recall_score(y_test, final_pred), 3),
        "f1": round(f1_score(y_test, final_pred), 3),
        "pr_auc": round(average_precision_score(y_test, final_prob), 3),
    },
}

Path("../models").mkdir(exist_ok=True)
joblib.dump(bundle, "../models/model.joblib")
print("Saved. Threshold:", round(final_threshold, 3), " Metrics:", bundle["metrics"])

loaded = joblib.load("../models/model.joblib")
same = np.allclose(loaded["pipeline"].predict_proba(X_test["text"])[:, 1], final_prob)
print("Reloaded model gives identical predictions:", same)

Saved. Threshold: 0.448  Metrics: {'precision': 0.804, 'recall': 0.872, 'f1': 0.837, 'pr_auc': 0.908}
Reloaded model gives identical predictions: True


In [26]:
import sys, sklearn
print(sys.executable)
print("scikit-learn version:", sklearn.__version__)

c:\Users\ravur\Desktop\fake-job-detector\venv\Scripts\python.exe
scikit-learn version: 1.9.1
